In [1]:
import pandas as pd
import numpy as np
import scipy
import scipy.sparse as sp
import scipy.io as sio
import scipy.stats as stats
from tqdm.notebook import tqdm


import os
os.environ["R_HOME"] = f"{os.environ['CONDA_PREFIX']}\\Lib\\R"


from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from plotnine import *
from mizani.formatters import label_number
fmt2 = label_number(accuracy=0.01)  # 2 decimal places

import matplotlib.pyplot as plt 


from scipy.sparse import coo_matrix
from scipy.sparse import csr_matrix
import pickle

from joblib import Parallel, delayed
from pathlib import Path
import sys

# Get project root as parent of notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.methods.GWASH_funcs import *
from src.methods.GWASH_sim_funcs import *
from src.methods.ldsc_barebones import *

from src.simtools.sim_utils import *
from src.simtools.data_generation import data_generation as data_generation
from src.simtools.data_preprocessing import data_preprocessing as data_preprocessing
from src.simtools.do_analysis import do_analysis as do_analysis
from src.simtools.run_simulations import run_simulations as run_simulations
from src.simtools.data_generation import gen_ref_ldscores_panel as gen_ref_ldscores_panel
from src.simtools.visualization import visualize

from natsort import natsorted

import pickle

os.chdir(PROJECT_ROOT)

# Two-step histograms

## AR1

In [2]:
# import pandas as pd
# import numpy as np
# import scipy
# import scipy.sparse as sp
# import scipy.io as sio
# import scipy.stats as stats
# from tqdm.notebook import tqdm


# import os
# import sys

# # Get the project root directory (one level above src)
# project_root = os.getcwd()

# # Add the project root directory to sys.path
# sys.path.append(project_root)

# if sys.platform == 'win32':
#     os.environ["R_HOME"] = f"{os.environ['CONDA_PREFIX']}\\Lib\\R"
#     n_jobs = 5
# else:
#     n_jobs = 5

# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import r2_score

# from plotnine import *

# import matplotlib.pyplot as plt 


# #import datatable as dt
# #from scipy.sparse import coo_matrix
# #from sparse_dot_mkl import dot_product_mkl

# from scipy.sparse import csr_matrix
# import pickle

# from joblib import Parallel, delayed

# from platform import python_version

# from src.methods.GWASH_funcs import *
# from src.methods.GWASH_sim_funcs import *
# from src.methods.ldsc_barebones import *

# from src.simtools.sim_utils import *
# from src.simtools.data_generation import data_generation as data_generation
# from src.simtools.data_preprocessing import data_preprocessing as data_preprocessing
# from src.simtools.do_analysis import do_analysis as do_analysis
# from src.simtools.run_simulations import run_simulations as run_simulations
# from src.simtools.data_generation import gen_ref_ldscores_panel as gen_ref_ldscores_panel


# from datetime import datetime

# #n,m,num_sims = int(sys.argv[1]),int(sys.argv[2]),int(sys.argv[3])
# n,m,num_sims = 5000,10000,100
# # Think about what parameters am I inputting:
# # Properties of X that can be tweaked: n,m, sigma_s, rho1, rho2, Fst
# # LD Matrix properties (if an actual LD matrix is used, its path, if a reference ldscore panel is generated): realistic, prefix,make_ref_ldscores
# # Method properties (related to methods used): reml_tol, reml_max_iters
# # Statistical properties: extra things like principal components regression: nPCs,regress_PC_out, regress_X_on_PC, regress_y_on_PC
# # Simulation Properties (related to running the actual simulations): num_sims
# # Debugging Properties (use this to actual check if the shortcuts we use are correct): calc_mu_hat_2_fast,old



# X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
# ld_mat_properties = {'realistic': False,'prefix': None,'make_ref_ldscores':False}
# ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
# method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
# stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
# simul_properties = {'num_sims':num_sims,'h2_pop': 0.2}
# debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress': True,'run_gcta': False}

# list_of_dicts = [X_properties,ld_mat_properties,ref_X_properties,method_properties,stats_properties,simul_properties,debug_properties]
# params  = combine_all_dicts(list_of_dicts)

# to_run = dict()
# to_run['not_realistic_ref_ldscores_once'] = params

# seed_num = 123

# for sim_key in tqdm(to_run.keys()):
    
#     locals().update(to_run[sim_key])
    
#     res_dict = dict()
#     res_dict_raw = dict()
    
#     #pm_causals = [0.005,0.05,0.5,0.75,float(1)]
#     pm_causals = [float(1)]
#     counter = -1
#     multithreading = True

    
#     for pm_causal in pm_causals:
#         counter += 1
#         np.random.seed(counter)
        
#         ### MAKE REF PANEL LIKE 1000 G
#         # meaning I generate ldscores separately and use this to try to predict
#         # using ref extremely different from X outside won't work

#         key = str(pm_causal)

#         if make_ref_ldscores:
#             ref_data_gen = data_generation(n = ref_n,m= m,Fst = ref_Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
#             ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_tilde_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
#             n_tilde = ref_data_gen.n
#         else:
#             ref_data_gen = None
#             ref_ldscores = None
#             n_tilde = None

#             ref_mu2_hat = None
#             ref_mu3_hat = None
#             X_tilde_ref = None
            
#         my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat, do_ldscore_bias_correct = True)
#         #res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress,run_gcta = run_gcta).num_sims_simulations_publication()
#         #res_dict_raw[key] = res_dfs
#         #res_dfs_tab = est_std_df(res_dfs)
#         #res_dict[key] = res_dfs_tab



# def run_a_toy_sim(my_data_gen,i,twostep = 30):
#     np.random.seed(seed_num + i)
#     X,b = my_data_gen.gen_X(realistic = False)
#     y = my_data_gen.gen_y(X,b)
    
    
#     my_data_preprocessing = data_preprocessing(my_data_gen,nPCs = 5,regress_X_on_PC = False,regress_y_on_PC = False)
#     if False:
#         regress_res = my_data_preprocessing.regress_out_PCs(X,y)
#         X = regress_res['X_res']
#         y = regress_res['y_res']
        
#     h2_samp_fve =  np.var(np.matmul(X,b),ddof=1)/np.var(y,ddof = 1)
#     if scaleX:
#         X_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(X)
#     else:
#         X_tilde = X
#     if scaley:
#         y_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(y)
#     else:
#         y_tilde = y
    
#     u2,s2,ldscores = my_data_gen.compute_u2_ldscores(X_tilde,y_tilde)

#     if my_data_gen.do_ldscore_bias_correct:
#         theoretical_rhs = ((ldscores * (n/m)).mean() * h2_pop) - 1
#     else:
#         theoretical_rhs = ((ldscores * (n/m) - 1).mean() * h2_pop) - 1

    
#     ldsc_reg_weights = ldscores
#     sims = do_analysis(my_data_gen,X_tilde,u2,s2,ldscores,ldsc_reg_weights,calc_mu_hat_2_fast = True)
#     #h2_ldsc_reg,se_ldsc_reg,icpt_ldsc_reg,weights = sims.do_ldsc()
#     h2_ldsc_reg,se_ldsc_reg,icpt_ldsc_reg,weights = do_ldsc_regression(u2,ldscores,my_data_gen.M,my_data_gen.N_per_SNP,ldsc_reg_weights,intercept = None,twostep = twostep)
    


#     sims_fixed = do_analysis(my_data_gen,X_tilde,u2,s2,ldscores,ldsc_reg_weights,intercept = 1,calc_mu_hat_2_fast = True)
#     h2_ldsc_fixed,se_ldsc_fixed,icpt_ldsc_fixed,weights = sims_fixed.do_ldsc()

#     gwash_res = sims.do_GWASH_from_ldscores()

#     return gwash_res['gwash'],h2_ldsc_reg,icpt_ldsc_reg,h2_ldsc_fixed,u2,ldscores,theoretical_rhs


# twosteps = [10,20,30,40,50]
# ldsc_free_res = np.zeros((num_sims,len(twosteps)))
# ldsc_free_intercept_res = np.zeros((num_sims,len(twosteps)))
# ldsc_fixed_res = np.zeros((num_sims,len(twosteps)))

# for z in range(len(twosteps)):
#     twostep = twosteps[z]
#     for i in tqdm(range(num_sims)):
#         gwash_h2, ldsc_free_h2,ldsc_free_intercept, ldsc_fixed_h2, u2, ldscores, theoretical_rhs = run_a_toy_sim(my_data_gen,i,twostep = twostep)
#         ldsc_free_res[i,z] = ldsc_free_h2
#         ldsc_free_intercept_res[i,z] = ldsc_free_intercept
#         ldsc_fixed_res[i,z] = ldsc_fixed_h2
    
# res = {'ldsc_free_res':ldsc_free_res,'ldsc_free_intercept_res':ldsc_free_intercept_res}
# with open("save_data/supplementary/AR1/ldsc_h2_intercept_twostep.pkl", "wb") as f:
#     pickle.dump(res, f)

In [3]:
res = pickle.load(open('save_data/supplementary/AR1/ldsc_h2_intercept_twostep.pkl','rb'))

ldsc_free_res = res['ldsc_free_res']
ldsc_free_intercept_res = res['ldsc_free_intercept_res']

In [4]:
# rows = sims 1:100
# columns -> twosteps = [10,20,30,40,50]

twosteps = [10,20,30,40,50]
df = []
mean_dfs = []
fpath = 'figures/supplementary/S1/'
os.makedirs(fpath,exist_ok = True)
for z in range(len(twosteps)):
    temp = pd.DataFrame(ldsc_free_intercept_res[:,z])
    temp['twostep'] = twosteps[z]
    temp.columns = ['intercept','twostep']

    df.append(temp)
    
    mean_df = temp.copy()
    mean_df = pd.DataFrame(mean_df.mean(axis = 0)).T
    
    mean_dfs.append(mean_df)

df = pd.concat(df,axis = 0)
df['twostep'] = pd.Categorical(df['twostep'])

mean_dfs = pd.concat(mean_dfs,axis = 0)
mean_dfs['twostep'] = pd.Categorical(mean_dfs['twostep'])

df1 = df[df['twostep'].isin([10,20,30,40,50])]


intercept_p = ggplot(df1,aes(x = 'intercept', y = after_stat('density'),fill = 'twostep')) + geom_histogram(alpha = 0.6, position = "identity") +geom_vline(xintercept = 1,color = 'black') + geom_vline(mean_dfs,aes(xintercept = 'intercept',color ='twostep'),linetype = 'dashed', show_legend=False) + theme(legend_position = 'bottom')

ggsave(intercept_p,fpath+'AR1_intercept_twostep.png',dpi=300)


In [5]:
# rows = sims 1:100
# columns -> twosteps = [10,20,30,40,50]
df = []
mean_dfs = []
for z in range(len(twosteps)):
    temp = pd.DataFrame(ldsc_free_res[:,z])
    temp['twostep'] = twosteps[z]
    temp.columns = ['ldsc_h2','twostep']
    df.append(temp)

    mean_df = temp.copy()
    mean_df = pd.DataFrame(mean_df.mean(axis = 0)).T
    
    mean_dfs.append(mean_df)

df = pd.concat(df,axis = 0)
df['twostep'] = pd.Categorical(df['twostep'])

mean_dfs = pd.concat(mean_dfs,axis = 0)
mean_dfs['twostep'] = pd.Categorical(mean_dfs['twostep'])

df1 = df[df['twostep'].isin([10,20,30,40,50])]

h2_p = ggplot(df1,aes(x = 'ldsc_h2', y = after_stat('density'),fill = 'twostep')) + geom_histogram(alpha = 0.6, position = "identity") +geom_vline(xintercept = 0.2,color = 'black') + geom_vline(mean_dfs,aes(xintercept = 'ldsc_h2',color ='twostep'),linetype = 'dashed', show_legend=False) + theme(legend_title = element_blank(),legend_position = 'bottom')

ggsave(h2_p,fpath+'AR1_h2_twostep.png',dpi=300)


In [6]:
legend_only = h2_p + theme_void() + theme(
     legend_position="bottom",  # Centers the legend
     figure_size=(10, 2)         # Compact size
 ) + ggtitle('')

legend_only += theme(legend_position=(0.5, 0.5),      # Places the center of legend at (50% x, 50% y)
     legend_direction='horizontal',   # Keeps the side-by-side look from your image
     legend_box_margin=0,
     legend_title=element_blank(),
     plot_margin=0) 
legend_only += coord_cartesian(xlim=(9998, 9999), ylim=(9998, 9999))

ggsave(legend_only,fpath + 'twostep_legend.png', dpi=300, width=5, height=0.5, bbox_inches='tight', pad_inches=0)

## Realistic

In [7]:
# import pandas as pd
# import numpy as np
# import scipy
# import scipy.sparse as sp
# import scipy.io as sio
# import scipy.stats as stats
# from tqdm.notebook import tqdm


# import os
# import sys

# # Get the project root directory (one level above src)
# project_root = os.getcwd()

# # Add the project root directory to sys.path
# sys.path.append(project_root)

# if sys.platform == 'win32':
#     os.environ["R_HOME"] = f"{os.environ['CONDA_PREFIX']}\\Lib\\R"
#     n_jobs = 5
# else:
#     n_jobs = 5

# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import r2_score

# from plotnine import *

# import matplotlib.pyplot as plt 


# from scipy.sparse import coo_matrix
# #from sparse_dot_mkl import dot_product_mkl

# from scipy.sparse import csr_matrix
# import pickle

# from joblib import Parallel, delayed

# from platform import python_version

# from src.methods.GWASH_funcs import *
# from src.methods.GWASH_sim_funcs import *
# from src.methods.ldsc_barebones import *

# from src.simtools.sim_utils import *
# from src.simtools.data_generation import data_generation as data_generation
# from src.simtools.data_preprocessing import data_preprocessing as data_preprocessing
# from src.simtools.do_analysis import do_analysis as do_analysis
# from src.simtools.run_simulations import run_simulations as run_simulations
# from src.simtools.data_generation import gen_ref_ldscores_panel as gen_ref_ldscores_panel


# from datetime import datetime

# #n,m,num_sims = int(sys.argv[1]),int(sys.argv[2]),int(sys.argv[3])
# n,m,num_sims = 5000,10000,100
# # Think about what parameters am I inputting:
# # Properties of X that can be tweaked: n,m, sigma_s, rho1, rho2, Fst
# # LD Matrix properties (if an actual LD matrix is used, its path, if a reference ldscore panel is generated): realistic, prefix,make_ref_ldscores
# # Method properties (related to methods used): reml_tol, reml_max_iters
# # Statistical properties: extra things like principal components regression: nPCs,regress_PC_out, regress_X_on_PC, regress_y_on_PC
# # Simulation Properties (related to running the actual simulations): num_sims
# # Debugging Properties (use this to actual check if the shortcuts we use are correct): calc_mu_hat_2_fast,old



# X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
# ld_mat_properties = {'realistic': True,'prefix': '1kg_p1_eur_ben/1kg_p1_eur_chr22','make_ref_ldscores':False}
# ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
# method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
# stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
# simul_properties = {'num_sims':num_sims,'h2_pop': 0.2}
# debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress': True,'run_gcta': False}

# list_of_dicts = [X_properties,ld_mat_properties,ref_X_properties,method_properties,stats_properties,simul_properties,debug_properties]
# params  = combine_all_dicts(list_of_dicts)

# to_run = dict()
# to_run['realistic_ref_ldscores_once'] = params

# seed_num = 123

# for sim_key in tqdm(to_run.keys()):
    
#     locals().update(to_run[sim_key])
    
#     res_dict = dict()
#     res_dict_raw = dict()
    
#     #pm_causals = [0.005,0.05,0.5,0.75,float(1)]
#     pm_causals = [float(1)]
#     counter = -1
#     multithreading = True

    
#     for pm_causal in pm_causals:
#         counter += 1
#         np.random.seed(counter)
        
#         ### MAKE REF PANEL LIKE 1000 G
#         # meaning I generate ldscores separately and use this to try to predict
#         # using ref extremely different from X outside won't work

#         key = str(pm_causal)

#         if make_ref_ldscores:
#             ref_data_gen = data_generation(n = ref_n,m= m,Fst = ref_Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
#             ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_tilde_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
#             n_tilde = ref_data_gen.n
#         else:
#             ref_data_gen = None
#             ref_ldscores = None
#             n_tilde = None

#             ref_mu2_hat = None
#             ref_mu3_hat = None
#             X_tilde_ref = None
            
#         my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat, do_ldscore_bias_correct = True)
#         #res_dfs = run_simulations(my_data_gen,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress,run_gcta = run_gcta).num_sims_simulations_publication()
#         #res_dict_raw[key] = res_dfs
#         #res_dfs_tab = est_std_df(res_dfs)
#         #res_dict[key] = res_dfs_tab



# def run_a_toy_sim(my_data_gen,i,twostep = 30):
#     np.random.seed(seed_num + i)
#     X,b = my_data_gen.gen_X(realistic = False)
#     y = my_data_gen.gen_y(X,b)
    
    
#     my_data_preprocessing = data_preprocessing(my_data_gen,nPCs = 5,regress_X_on_PC = False,regress_y_on_PC = False)
#     if False:
#         regress_res = my_data_preprocessing.regress_out_PCs(X,y)
#         X = regress_res['X_res']
#         y = regress_res['y_res']
        
#     h2_samp_fve =  np.var(np.matmul(X,b),ddof=1)/np.var(y,ddof = 1)
#     if scaleX:
#         X_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(X)
#     else:
#         X_tilde = X
#     if scaley:
#         y_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(y)
#     else:
#         y_tilde = y
    
#     u2,s2,ldscores = my_data_gen.compute_u2_ldscores(X_tilde,y_tilde)

#     if my_data_gen.do_ldscore_bias_correct:
#         theoretical_rhs = ((ldscores * (n/m)).mean() * h2_pop) - 1
#     else:
#         theoretical_rhs = ((ldscores * (n/m) - 1).mean() * h2_pop) - 1

    
#     ldsc_reg_weights = ldscores
#     sims = do_analysis(my_data_gen,X_tilde,u2,s2,ldscores,ldsc_reg_weights,calc_mu_hat_2_fast = True)
#     #h2_ldsc_reg,se_ldsc_reg,icpt_ldsc_reg,weights = sims.do_ldsc()
#     h2_ldsc_reg,se_ldsc_reg,icpt_ldsc_reg,weights = do_ldsc_regression(u2,ldscores,my_data_gen.M,my_data_gen.N_per_SNP,ldsc_reg_weights,intercept = None,twostep = twostep)
    


#     sims_fixed = do_analysis(my_data_gen,X_tilde,u2,s2,ldscores,ldsc_reg_weights,intercept = 1,calc_mu_hat_2_fast = True)
#     h2_ldsc_fixed,se_ldsc_fixed,icpt_ldsc_fixed,weights = sims_fixed.do_ldsc()

#     gwash_res = sims.do_GWASH_from_ldscores()

#     return gwash_res['gwash'],h2_ldsc_reg,icpt_ldsc_reg,h2_ldsc_fixed,u2,ldscores,theoretical_rhs


# twosteps = [10,20,30,40,50]
# ldsc_free_res = np.zeros((num_sims,len(twosteps)))
# ldsc_free_intercept_res = np.zeros((num_sims,len(twosteps)))
# ldsc_fixed_res = np.zeros((num_sims,len(twosteps)))

# for z in range(len(twosteps)):
#     twostep = twosteps[z]
#     for i in tqdm(range(num_sims)):
#         gwash_h2, ldsc_free_h2,ldsc_free_intercept, ldsc_fixed_h2, u2, ldscores, theoretical_rhs = run_a_toy_sim(my_data_gen,i,twostep = twostep)
#         ldsc_free_res[i,z] = ldsc_free_h2
#         ldsc_free_intercept_res[i,z] = ldsc_free_intercept
#         ldsc_fixed_res[i,z] = ldsc_fixed_h2
    


In [8]:
# res = {'ldsc_free_res':ldsc_free_res,'ldsc_free_intercept_res':ldsc_free_intercept_res}
# with open("save_data/supplementary/realistic/ldsc_h2_intercept_twostep.pkl", "wb") as f:
#     pickle.dump(res, f)

In [9]:
res = pickle.load(open('save_data/supplementary/realistic/ldsc_h2_intercept_twostep.pkl','rb'))

ldsc_free_res = res['ldsc_free_res']
ldsc_free_intercept_res = res['ldsc_free_intercept_res']

In [10]:
# rows = sims 1:100
# columns -> twosteps = [1,10,20,30,40,50]
df = []
mean_dfs = []
fpath = 'figures/supplementary/S1/'
os.makedirs(fpath,exist_ok = True)
for z in range(len(twosteps)):
    temp = pd.DataFrame(ldsc_free_intercept_res[:,z])
    temp['twostep'] = twosteps[z]
    temp.columns = ['intercept','twostep']

    df.append(temp)
    
    mean_df = temp.copy()
    mean_df = pd.DataFrame(mean_df.mean(axis = 0)).T
    
    mean_dfs.append(mean_df)

df = pd.concat(df,axis = 0)
df['twostep'] = pd.Categorical(df['twostep'])

mean_dfs = pd.concat(mean_dfs,axis = 0)
mean_dfs['twostep'] = pd.Categorical(mean_dfs['twostep'])

df1 = df[df['twostep'].isin([10,20,30,40,50])]


intercept_p = ggplot(df1,aes(x = 'intercept', y = after_stat('density'),fill = 'twostep')) + geom_histogram(alpha = 0.6, position = "identity") +geom_vline(xintercept = 1,color = 'black') + geom_vline(mean_dfs,aes(xintercept = 'intercept',color ='twostep'),linetype = 'dashed', show_legend=False) + theme(legend_title = element_blank(),legend_position = 'bottom')

ggsave(intercept_p,fpath+'realistic_intercept_twostep.png',dpi=300)

In [11]:
# rows = sims 1:100
# columns -> twosteps = [1,10,20,30,40,50]
df = []
mean_dfs = []
for z in range(len(twosteps)):
    temp = pd.DataFrame(ldsc_free_res[:,z])
    temp['twostep'] = twosteps[z]
    temp.columns = ['ldsc_h2','twostep']
    df.append(temp)

    mean_df = temp.copy()
    mean_df = pd.DataFrame(mean_df.mean(axis = 0)).T
    
    mean_dfs.append(mean_df)

df = pd.concat(df,axis = 0)
df['twostep'] = pd.Categorical(df['twostep'])

mean_dfs = pd.concat(mean_dfs,axis = 0)
mean_dfs['twostep'] = pd.Categorical(mean_dfs['twostep'])

df1 = df[df['twostep'].isin([10,20,30,40,50])]

h2_p = ggplot(df1,aes(x = 'ldsc_h2', y = after_stat('density'),fill = 'twostep')) + geom_histogram(alpha = 0.6, position = "identity") +geom_vline(xintercept = 0.2,color = 'black') + geom_vline(mean_dfs,aes(xintercept = 'ldsc_h2',color ='twostep'),linetype = 'dashed', show_legend=False) + theme(legend_title = element_blank(),legend_position = 'bottom')

ggsave(h2_p,fpath+'realistic_h2_twostep.png',dpi=300)

# Bias Correction Justification

This notebook serves as an explanation of why $\hat{\mu}_{2}$ and $\hat{\mu}_{3}$ must be computed differently that in Schwartzman et al 2019. The main reason is because the bias correction done in Bulik-Sulivan et al 2015 is different than the terms done in the original GWASH paper. Here, realistic LD is used to show the differences between implementations.

In [12]:
# res = dict()
# for n in [100,200,300,378,400,500,1000,2000,3000,4000,5000]:
#     X_properties = {'n': n,'m':10000,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
#     ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
#     ld_mat_properties = {'realistic': True,'model_Fst_in_realistic':True,'prefix': '1kg_p1_eur_ben/1kg_p1_eur_chr22','make_ref_ldscores':True}
#     method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
#     stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
#     simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
#     debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}
    
#     list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
#     params  = combine_all_dicts(list_of_dicts)
    
#     to_run = dict()
#     to_run['demonstration'] = params
    
#     sim_key = 'demonstration'
#     locals().update(to_run[sim_key])
#     res_dict = dict()
#     res_dict_raw = dict()
    
#     #rhos = [0.995]
#     counter = -1
#     multithreading = True
    
    
#     counter += 1
#     Fsts = [0,0.01,0.03,0.05,0.07,0.1]
#     Fst = 0
    
    
#     n_jobs = 1
    
#     C = 0
#     C_vec = np.concatenate([np.repeat(C,5000),np.repeat(-C,5000)]).reshape(-1,1)
    
    
#     res_dfs_ref_panel_C = []
#     num_sims = 100
    
#     mean_ldscores_nbc = []
#     mean_ldscores = []
#     mu2_hat_olds = []
#     mu2_hats = []

#     my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,model_Fst_in_realistic = model_Fst_in_realistic,realistic = realistic,prefix = prefix,ref_ldscores = None,ref_mu2_hat = None,ref_mu3_hat = None)
#     my_data_gen_no_bias_correct = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,model_Fst_in_realistic = model_Fst_in_realistic,realistic = realistic,prefix = prefix,ref_ldscores = None,ref_mu2_hat = None,ref_mu3_hat = None,do_ldscore_bias_correct = False)
    
#     for i in tqdm(range(num_sims)):
#         np.random.seed(seed_num + i)
        
#         X,b = my_data_gen.gen_X(realistic = realistic)
#         y = my_data_gen.gen_y(X,b)
        
        
#         my_data_preprocessing = data_preprocessing(my_data_gen,nPCs = nPCs,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC)
        
#         if regress_PC_out:
#             regress_res = my_data_preprocessing.regress_out_PCs(X,y)
#             X = regress_res['X_res']
#             y = regress_res['y_res']
            
#         h2_samp_fve =  np.var(np.matmul(X,b),ddof=1)/np.var(y,ddof = 1)
#         if scaleX:
#             X_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(X)
#         else:
#             X_tilde = X
#         if scaley:
#             y_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(y)
#         else:
#             y_tilde = y
        
        
#         # if my_data_gen.ref_ldscores is None, then generate ldscores. Otherwise, use my_data_gen.ref_ldscores
        
#         u2,s2,ldscores_no_bias_correct = my_data_gen_no_bias_correct.compute_u2_ldscores(X_tilde,y_tilde) # Bias Corrected LD Scores as written in Bulik-Sulivan et al 2015.
#         mean_ldscores_nbc.append(ldscores_no_bias_correct.mean())
#         u2,s2,ldscores = my_data_gen.compute_u2_ldscores(X_tilde,y_tilde) # Bias Corrected LD Scores as written in Bulik-Sulivan et al 2015.
#         mean_ldscores.append(ldscores.mean())
#         ldsc_reg_weights = ldscores
#         sims = do_analysis(my_data_gen,X_tilde,u2,s2,ldscores,ldsc_reg_weights,calc_mu_hat_2_fast = True)
#         gwash_res_sample_theoretical = sims.do_GWASH_from_X_tilde() # Do same calculation from X_tilde() just to get theoretical variance.
        
#         # As written exactly from 2019 paper
#         gwash_mu2_hat_old = gwash_res_sample_theoretical['mu2_hat_old']
#         mu2_hat_olds.append(gwash_mu2_hat_old)
#         gwash_mu3_hat_old = gwash_res_sample_theoretical['mu3_hat_old']
        
#         # With new bias-correction terms to make approximate to Bulik-Sulivan
#         gwash_mu2_hat = gwash_res_sample_theoretical['mu2_hat']
#         mu2_hats.append(gwash_mu2_hat)
#         gwash_mu3_hat = gwash_res_sample_theoretical['mu3_hat']
#     res[str(n)] = pd.DataFrame({'ldscores_no_bias_correct':mean_ldscores_nbc,'mu2_hat_2019':mu2_hat_olds,'ldscores':mean_ldscores,'mu2_hat':mu2_hats})
#f = open("save_data/supplementary/realistic/mu2_hat_comparison.pkl","wb")
#pickle.dump(res,f)
#f.close()

with open("save_data/supplementary/realistic/mu2_hat_comparison.pkl", 'rb') as file:
    # Load the pickled data
    res = pickle.load(file)

fpath = 'figures/supplementary/S2/'
os.makedirs(fpath,exist_ok = True)

df_to_plot = []
for key in res.keys():
    temp = pd.DataFrame(pd.DataFrame(1 - res[key]['mu2_hat']/res[key]['mu2_hat_2019']).mean(axis = 0))
    temp.columns = ['mean_relative_difference']
    temp['n'] = int(key)
    temp['type'] = 'mu2_hat_2019'
    df_to_plot.append(temp)
df_to_plot = pd.concat(df_to_plot,axis = 0).reset_index(drop = True)

p =ggplot(df_to_plot,aes(x = 'n',y = 'mean_relative_difference',color = 'type')) + geom_point() + scale_color_manual('black',labels=[r'$\hat{\mu}_{2}$']) + geom_vline(xintercept = 378,linetype = '--',color = 'red') + ylab(r'Mean Relative Difference')
ggsave(p,fpath+'mu2_hat_comparison.png',dpi=300)

df_to_plot = []
for key in res.keys():
    temp = pd.DataFrame(pd.DataFrame(1-res[key]['mu2_hat_2019']/res[key]['ldscores']).mean(axis = 0))
    temp.columns = ['mean_relative_difference']
    temp['n'] = int(key)
    temp['type'] = 'mu2_hat_2019'
    df_to_plot.append(temp)
for key in res.keys():
    temp = pd.DataFrame(pd.DataFrame(1-res[key]['mu2_hat']/res[key]['ldscores']).mean(axis = 0))
    temp.columns = ['mean_relative_difference']
    temp['n'] = int(key)
    temp['type'] = 'mu2_hat'
    df_to_plot.append(temp)
df_to_plot = pd.concat(df_to_plot,axis = 0).reset_index(drop = True)


p = ggplot(df_to_plot,aes(x = 'n',y = 'mean_relative_difference',color = 'type')) + geom_point() + geom_vline(xintercept = 378,linetype = '--',color = 'red') + ylab(r'Mean Relative Difference') + scale_color_discrete(labels=[r'$\tilde{\mu}_{2}$',r'$\hat{\mu}_{2}$'])
ggsave(p,fpath+'mu2_hat_bc_justification.png',dpi=300)


# GWASH SE from true AR1 correlation structure vs empirical AR1 correlation structure from data

This figure shows the standard error of GWASH from $\mu_{2}$ and $\mu_{3}$ constructed from the true AR1 correlation structure against $\hat{\mu}_{2}$ and $\hat{\mu}_{3}$ constructed from $\tilde{S}$.

In [13]:
# n = 5000
# m = 10000
# h2_pop = 0.2
# rhos = [0,0.4,0.8,0.9,0.95,0.99,0.995]
# mu2s = []
# mu3s = []
# gwash_ses = []
# for rho in rhos:
#     res = calc_true_gwash_var(n,m,rho,h2_pop)
#     mu2s.append(res['mu2'])
#     mu3s.append(res['mu3'])
#     gwash_ses.append(res['gwash_se'])
# res_df = pd.DataFrame([np.repeat(n,len(rhos)),np.repeat(m,len(rhos)),rhos,mu2s,mu3s,gwash_ses]).T
# res_df.columns = ['n','m','rho','mu2','mu3','gwash_se']
# res_df.to_csv('save_data/supplementary/AR1/AR1_true_se.txt')

In [14]:
res_dict_raw = pkl_file_loader('save_data/main_text/AR1/single_param/rho.pkl')

## SUPPLEMENTARY WHERE I COMPARE NON REALISTIC RHO EMP SE vs TRUE SE
sim_gwash = []
for k in res_dict_raw.keys():
    sim_gwash.append(res_dict_raw[k]['h2_gwash'].std(ddof = 1))

true_AR1_GWASH_se = pd.read_table('save_data/supplementary/AR1/AR1_true_se.txt')
true_AR1_GWASH_se['emp_gwash_se'] = sim_gwash

temp1 = true_AR1_GWASH_se[['rho','gwash_se']]
temp1.columns = ['rho','se']
temp1['type'] = 'true gwash se'

temp2 = true_AR1_GWASH_se[['rho','emp_gwash_se']]
temp2.columns = ['rho','se']
temp2['type'] = 'empirical gwash se'
gwash_se_to_plot = pd.concat([temp1,temp2],axis = 0)
gwash_se_to_plot['rho'] = pd.Categorical(gwash_se_to_plot['rho'])
#color_legend = ["purple","green",'#FB93C4',"red"]
#color_obj = scale_color_manual(color_legend,breaks = breaks_legend,labels = ['GCTA','GWASH','LDSC Constrained Intercept','LDSC Free Intercept'])
title = r'AR1 $\rho$ True GWASH se vs. Empirical GWASH se' 

p = ggplot(gwash_se_to_plot,aes(x = 'rho',y = 'se',color = 'type')) + geom_point(alpha = 0.65) + scale_color_manual(['green','#98ff98']) + ggtitle(title) + xlab(r'$\rho$')
fpath = 'figures/supplementary/S3/'
os.makedirs(fpath,exist_ok = True)
ggsave(p,fpath+'true_vs_emp_gwash_se.png',dpi=300)

# $\textrm{prop}_\textrm{causal}$ - AR1

In [15]:
# LOAD ref level pm_causal results
res_dict_raw = pkl_file_loader('save_data/main_text/AR1/single_param/pm_causal.pkl')
param = r'$\mathrm{prop_{causal}}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))


p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels  

fig_path = 'figures/supplementary/S4/'
os.makedirs(fig_path,exist_ok = True)
ggsave(p,fig_path + 'non_realistic_ref_panel_pm_causal.png', dpi=300)


param = r'$\mathrm{prop_{causal}}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

fig_path = 'figures/supplementary/S4/'
os.makedirs(fig_path,exist_ok = True)
for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/AR1/double_param/Fst'+str(Fst).replace('.','')+'pm_causal.pkl')
    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,legend_position = 'none').plot_lineplot() + axis_labels
    ggsave(p,fig_path + 'not_realistic_ref_panel_Fst{Fst}_pm_causal.png'.format(Fst = str(Fst).replace('.','')), dpi=300)


res_dict_raw = pkl_file_loader('save_data/main_text/AR1/single_param/pm_causal.pkl')

param = r'$\mathrm{prop_{causal}}$'

fig_path = 'figures/supplementary/S6/'
os.makedirs(fig_path,exist_ok = True)


p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').make_plot_seplot_prop()
ggsave(p,fig_path + 'not_realistic_ref_panel_se_pm_causal_all_estimators_logpropplot.png', dpi= 600)


fig_path = 'figures/supplementary/S6/'
os.makedirs(fig_path,exist_ok = True)



for Fst in [0.05,0.1]:
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').make_plot_seplot_prop()
    ggsave(p,fig_path + 'not_realistic_ref_panel_se_Fst{Fst}_pm_causal_all_estimators_logpropplot.png'.format(Fst = str(Fst).replace('.','')), dpi=300)

legend_only = p + theme_void() + theme(
     legend_position="bottom",  # Centers the legend
     figure_size=(10, 2)         # Compact size
 )

legend_only += theme(legend_position=(0.5, 0.5),      # Places the center of legend at (50% x, 50% y)
     legend_direction='horizontal',   # Keeps the side-by-side look from your image
     legend_box_margin=0,
     legend_title=element_blank(),
     plot_margin=0) 
legend_only += coord_cartesian(xlim=(9998, 9999), ylim=(9998, 9999))


ggsave(legend_only,fig_path + 'legend.png'.format(Fst = str(Fst).replace('.','')), dpi=300, width=5, height=0.5, bbox_inches='tight', pad_inches=0)

fig_path = 'figures/supplementary/S8/'
os.makedirs(fig_path,exist_ok = True)

res_dict_raw = pkl_file_loader('save_data/main_text/AR1/single_param/pm_causal.pkl')


param = r'$\mathrm{prop}_{causal}$'

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').plot_studies_passed()

ggsave(p,fig_path + 'studies_passed_non_realistic_ref_pm_causal.png', dpi=300)

fig_path = 'figures/supplementary/S8/'
os.makedirs(fig_path,exist_ok = True)
param = r'$\mathrm{prop}_{causal}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))


for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/AR1/double_param/Fst'+str(Fst).replace('.','')+'pm_causal.pkl')
    
    param = r'$\mathrm{prop_{causal}}$'

    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all',legend_position = 'none').plot_studies_passed()
    ggsave(p,fig_path + 'studies_passed_non_realistic_ref_Fst{Fst}_pm_causal.png'.format(Fst = str(Fst)), dpi=300)



# $\textrm{prop}_\textrm{causal}$ - Realistic

In [16]:
# LOAD individual level pm_causal results
res_dict_raw = pkl_file_loader('save_data/main_text/realistic/single_param/pm_causal.pkl')
param = r'$\mathrm{prop_{causal}}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))


p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,legend_position = 'none').plot_lineplot() + axis_labels  

fig_path = 'figures/supplementary/S5/'
os.makedirs(fig_path,exist_ok = True)

ggsave(p,fig_path + 'realistic_ref_panel_pm_causal.png', dpi=300)


param = r'$\mathrm{prop}_{causal}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated LD X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))


fig_path = 'figures/supplementary/S5/'
os.makedirs(fig_path,exist_ok = True)

for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/realistic/double_param/Fst'+str(Fst).replace('.','')+'pm_causal.pkl')
    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels 
    ggsave(p,fig_path + 'realistic_ref_panel_Fst{Fst}_pm_causal.png'.format(Fst = str(Fst).replace('.','')), dpi=300)


res_dict_raw = pkl_file_loader('save_data/main_text/realistic/single_param/pm_causal.pkl')

param = r'$\mathrm{prop_{causal}}$'



fig_path = 'figures/supplementary/S7/'
os.makedirs(fig_path,exist_ok = True)


p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').make_plot_seplot_prop()
ggsave(p,fig_path + 'realistic_ref_panel_se_pm_causal_all_estimators_logpropplot.png', dpi=300)


fig_path = 'figures/supplementary/S7/'
os.makedirs(fig_path,exist_ok = True)
for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/realistic/double_param/Fst'+str(Fst).replace('.','')+'pm_causal.pkl')
    
    param = r'$\mathrm{prop}_{causal}$'
    

    #combined_dict[str(Fst)] = combined_dict1
    #res_focal = combined_dict[str(Fst)]
    #p = visualize(res_focal,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels 
    #ggsave(p,'figures/' + 'fig7_Fst{Fst}_pm_causal.png'.format(Fst = str(Fst).replace('.','')), dpi=300)

    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').make_plot_seplot_prop()
    ggsave(p,fig_path + 'realistic_ref_panel_se_Fst{Fst}_pm_causal_all_estimators_logpropplot.png'.format(Fst = str(Fst).replace('.','')), dpi=300)
    

fig_path = 'figures/supplementary/S9/'
os.makedirs(fig_path,exist_ok = True)

param = r'$\mathrm{prop}_{causal}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated LD X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))


res_dict_raw = pkl_file_loader('save_data/main_text/realistic/single_param/pm_causal.pkl')


param = r'$\mathrm{prop}_{causal}$'

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').plot_studies_passed()

ggsave(p,fig_path + 'studies_passed_realistic_ref_pm_causal.png', dpi=300)

for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/realistic/double_param/Fst'+str(Fst).replace('.','')+ 'pm_causal.pkl')

    param = r'$\mathrm{prop_{causal}}$'
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').plot_studies_passed()
    ggsave(p,fig_path + 'studies_passed_realistic_ref_Fst{Fst}_pm_causal.png'.format(Fst = str(Fst).replace('.','')), dpi=300)


# AR1 Simulations with no Reference Panel

In [17]:
fig_path = 'figures/supplementary/S10/'
os.makedirs(fig_path,exist_ok = True)

res_dict_raw = pkl_file_loader('save_data/supplementary/AR1/single_param/rho.pkl')
param = r'$\rho$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels

ggsave(p,fig_path + 'non_realistic_rho.png', dpi=300)


param = r'$\rho$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))


for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/supplementary/AR1/double_param/Fst'+str(Fst).replace('.','')+'rho.pkl')
    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels
    ggsave(p,fig_path + 'not_realistic_Fst{Fst}_rho.png'.format(Fst = str(Fst).replace('.','')), dpi=300)

# LOAD individual level sigma_s results for GCTA
param_interest = 'sigma_s'
res_dict_raw = pkl_file_loader('save_data/supplementary/AR1/single_param/sigma_s.pkl')
param = r'$\sigma_{s}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels


ggsave(p,fig_path + 'non_realistic_sigma_s.png', dpi=300)

param = r'$\sigma_{s}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))


for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/supplementary/AR1/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl')
    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels
    ggsave(p,fig_path + 'not_realistic_Fst{Fst}_sigma_s.png'.format(Fst = str(Fst).replace('.','')), dpi=300)

# Realistic Simulations with no Reference Panel

Realistic LD Structure from Chromosome 22 and changing $\mathrm{prop_{causal}}$, $\sigma_{s}$, $\mathrm{F_{st}}$

## Changing One Parameter at a Time

In [18]:
fig_path = 'figures/supplementary/S11/'
os.makedirs(fig_path,exist_ok = True)


# LOAD individual level pm_causal results
res_dict_raw = pkl_file_loader('save_data/supplementary/realistic/single_param/sigma_s.pkl')
param = r'$\sigma_{s}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))


p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels


ggsave(p,fig_path + 'realistic_sigma_s.png', dpi=300)


param = r'$\sigma_{s}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated LD X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/supplementary/realistic/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl')
    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels
    ggsave(p,fig_path + 'realistic_Fst{Fst}_sigma_s.png'.format(Fst = str(Fst).replace('.','')), dpi=300)


# Estimation of standard error in AR1 Simulations


In [19]:
fig_path = 'figures/supplementary/S12/'
os.makedirs(fig_path,exist_ok = True)

res_dict_raw = pkl_file_loader('save_data/supplementary/AR1/single_param/rho.pkl')
param = r'$\rho$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).make_plot_seplot_prop() + axis_labels

ggsave(p,fig_path + 'non_realistic_se_rho_logpropplot.png', dpi=300)


param = r'$\rho$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))


for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/supplementary/AR1/double_param/Fst'+str(Fst).replace('.','')+'rho.pkl')
    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).make_plot_seplot_prop() + axis_labels
    ggsave(p,fig_path + 'not_realistic_se_Fst{Fst}_rho_logpropplot.png'.format(Fst = str(Fst).replace('.','')), dpi=300)

# LOAD individual level sigma_s results for GCTA
param_interest = 'sigma_s'
res_dict_raw = pkl_file_loader('save_data/supplementary/AR1/single_param/sigma_s.pkl')
param = r'$\sigma_{s}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).make_plot_seplot_prop() + axis_labels


ggsave(p,fig_path + 'not_realistic_se_sigma_s_logpropplot.png', dpi=300)

param = r'$\sigma_{s}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))


for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/supplementary/AR1/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl')
    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).make_plot_seplot_prop() + axis_labels
    ggsave(p,fig_path + 'not_realistic_se_Fst{Fst}_sigma_s_logpropplot.png'.format(Fst = str(Fst).replace('.','')), dpi=300)

# Estimation of standard error in Realistic LD Simulations

In [20]:
fig_path = 'figures/supplementary/S13/'
os.makedirs(fig_path,exist_ok = True)

res_dict_raw = pkl_file_loader('save_data/supplementary/realistic/single_param/sigma_s.pkl')

param = r'$\sigma_{s}$'

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all',legend_position = 'none').make_plot_seplot_prop()

ggsave(p,fig_path + 'realistic_se_sigma_s_all_estimators.png', dpi=300)

for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/supplementary/realistic/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl')
    
    param = r'$\sigma_{s}$'
    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').make_plot_seplot_prop()

    ggsave(p,fig_path + 'realistic_se_Fst{Fst}_sigma_s_all_estimators.png'.format(Fst = str(Fst).replace('.','')), dpi=300)
    
    #combined_dict[str(Fst)] = combined_dict1
    #res_focal = combined_dict[str(Fst)]
    #p = visualize(res_focal,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels + title
    #ggsave(p,'figures/supplementary/' + 'fig7_Fst{Fst}_pm_causal.png'.format(Fst = str(Fst).replace('.','')), dpi=300)


# Impact of Z-scores on AR1 Simulations

In [21]:
fig_path = 'figures/supplementary/S14/'
os.makedirs(fig_path,exist_ok = True)

res_dict_raw = pkl_file_loader('save_data/supplementary/AR1/single_param/rho.pkl')
param = r'$\rho$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_studies_passed()

ggsave(p,fig_path + 'studies_passed_non_realistic_rho.png', dpi=300)


param = r'$\rho$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))


for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/supplementary/AR1/double_param/Fst'+str(Fst).replace('.','')+'rho.pkl')
    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_studies_passed()
    ggsave(p,fig_path + 'studies_passed_not_realistic_Fst{Fst}_rho.png'.format(Fst = str(Fst).replace('.','')), dpi=300)

# LOAD individual level sigma_s results for GCTA
param_interest = 'sigma_s'
res_dict_raw = pkl_file_loader('save_data/supplementary/AR1/single_param/sigma_s.pkl')
param = r'$\sigma_{s}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_studies_passed()


ggsave(p,fig_path + 'studies_passed_non_realistic_sigma_s.png', dpi=300)

param = r'$\sigma_{s}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))


for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/supplementary/AR1/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl')
    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_studies_passed()
    ggsave(p,fig_path + 'studies_passed_not_realistic_Fst{Fst}_sigma_s.png'.format(Fst = str(Fst).replace('.','')), dpi=300)

# Impact of Z-scores on Realistic LD Simulations

In [22]:
fig_path = 'figures/supplementary/S15/'
os.makedirs(fig_path,exist_ok = True)

res_dict_raw = pkl_file_loader('save_data/supplementary/realistic/single_param/sigma_s.pkl')

param = r'$\sigma_{s}$'

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all',legend_position = 'none').plot_studies_passed()

ggsave(p,fig_path + 'studies_passed_realistic_sigma_s.png', dpi=300)

for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/supplementary/realistic/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl')
    
    param = r'$\sigma_{s}$'
    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').plot_studies_passed()

    ggsave(p,fig_path + 'studies_passed_realistic_Fst{Fst}_sigma_s.png'.format(Fst = str(Fst).replace('.','')), dpi=300)
    
    #combined_dict[str(Fst)] = combined_dict1
    #res_focal = combined_dict[str(Fst)]
    #p = visualize(res_focal,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels + title
    #ggsave(p,'figures/supplementary/' + 'fig7_Fst{Fst}_pm_causal.png'.format(Fst = str(Fst).replace('.','')), dpi=300)
